## Games Data Load

In [1]:
import requests
import mysql.connector
from mysql.connector import Error

In [2]:
try:
    conn = mysql.connector.connect(
        host="localhost", database="nhl_analytics_hub", user="root", password="root"
    )
    if conn.is_connected:
        print("Connected successfully...")
    cur = conn.cursor()
    cur.execute("SELECT team_abbrev, team_id FROM teams")
    team_id_map = dict(cur.fetchall())
    print(team_id_map)
    SEASON = "20252026"
    for abbrev, team_id in team_id_map.items():
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{abbrev}/20252026"
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        schedule = resp.json()

        games = schedule.get("games")

        for g in games:
            home_abbrev = g["homeTeam"]["abbrev"]
            away_abbrev = g["awayTeam"]["abbrev"]

            cur.execute(
                """
                INSERT IGNORE INTO games (
                    game_id, season, game_type, game_date, home_team_id,
                    away_team_id, home_score, away_score, game_state, venue_name
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    g["id"],
                    SEASON,
                    g.get("gameType"),
                    g.get("gameDate"),
                    team_id_map[home_abbrev],
                    team_id_map[away_abbrev],
                    g.get("home_score") or 0,
                    g.get("away_score") or 0,
                    g.get("gameState"),
                    g["venue"].get("default"),
                ),
            )

    conn.commit()

except Error as e:
    print("Exception Occurred...")
    print(e)

finally:
    if conn.is_connected:
        cur.close()
        conn.close()

Connected successfully...
{'ANA': 20, 'BOS': 11, 'BUF': 19, 'CAR': 1, 'CBJ': 29, 'CGY': 21, 'CHI': 28, 'COL': 10, 'DAL': 8, 'DET': 26, 'EDM': 3, 'FLA': 13, 'LAK': 30, 'MIN': 2, 'MTL': 9, 'NJD': 6, 'NSH': 12, 'NYI': 15, 'NYR': 18, 'OTT': 4, 'PHI': 22, 'PIT': 31, 'SEA': 24, 'SJS': 23, 'STL': 16, 'TBL': 7, 'TOR': 27, 'UTA': 17, 'VAN': 32, 'VGK': 5, 'WPG': 14, 'WSH': 25}
